# 01 · Define & Explore — enzyme design, ester hydrolysis, and the triad theozyme

**Standard slot:** *define & explore.* **For Project 21 this means:** understand de novo enzyme
design and the **serine-hydrolase mechanism**, then **construct the theozyme** (the Ser-His-Asp
triad + the oxyanion hole around the ester tetrahedral intermediate) and run a mock
theozyme→scaffold hello-world, including the catalytic-Ser→Ala dead-mutant control (D0).

Run `00_setup.ipynb` first in this session.

## Why serine hydrolases are a great de novo enzyme target
Serine hydrolases — esterases, lipases, proteases — are the **industrial workhorses** of
biocatalysis (ester synthesis, kinetic resolution of chiral building blocks, detergents). They
all run on the **Ser-His-Asp triad + oxyanion hole**:
- the catalytic **His** (oriented by **Asp/Glu**) deprotonates the **Ser** hydroxyl;
- **Ser-OG attacks the ester carbonyl C**, forming a **tetrahedral intermediate** whose oxyanion
  is stabilised by the **oxyanion hole** (two backbone-amide NHs);
- the intermediate collapses to an acyl-enzyme, which a His-activated water hydrolyses.

The honest history: for decades you could only *borrow and tweak* a natural hydrolase. The **2025
Science work (Lauko et al.)** *designed efficient hydrolases from scratch* — the tractable,
validated frontier result this project reproduces on a general esterase.

## The theozyme — the catalytic motif you must build
A **theozyme** ("theoretical enzyme") is the minimal set of catalytic functional groups placed
around the **tetrahedral intermediate**. For ester hydrolysis the canonical motif is:

| Role | Residue(s) | Job in the TS |
|------|-----------|----------------|
| catalytic Ser | Ser | Ser-OG attacks the ester carbonyl C (the nucleophile) |
| catalytic His | His | deprotonates Ser-OH (general base) |
| catalytic Asp | Asp / Glu | orients & protonates His (charge-relay) |
| oxyanion hole | 2 × backbone-NH (or Ser-OG) | stabilise the developing oxyanion |

You **construct** this from the literature and/or a QM tetrahedral-intermediate model — it is a
teaching template (`data/inputs/theozyme_def.txt`), **not** fabricated experimental data. *Do not
forget the oxyanion hole* — it is easy to omit and decisive for catalysis.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Build the theozyme (mock hello-world)
`scripts/enzyme_tools.py` exposes `build_theozyme(reaction)` → a functional-group geometry spec.
The distances/angles it ships are **PLACEHOLDERS** — replace them in `data/inputs/theozyme_def.txt`
(and in `build_theozyme`) with real, cited values during P1. This is the **enzyme-family template**
(Project 18 Kemp): here we override the motif with the Ser-His-Asp triad + oxyanion hole.

In [ ]:
from enzyme_tools import build_theozyme

theo = build_theozyme("ester_hydrolysis")
print("Reaction :", theo.reaction)
print("Substrate:", theo.substrate)
print("Provenance:", theo.provenance)
print("\nCatalytic functional groups (PLACEHOLDER geometry — fill from literature/QM):")
for fg in theo.functional_groups:
    ang = f"{fg.target_angle}deg" if fg.target_angle is not None else "n/a"
    print(f"  {fg.role:16s} {fg.residue}/{fg.atom:4s}  d={fg.target_distance}A  angle={ang}")
print("\nCatalytic residues to FIX during sequence design:", theo.catalytic_residue_ids())

## A first mock scaffold + sequence + the dead-mutant control (no GPU)
`scaffold_motif(...)` (mock) returns placeholder backbones presenting the motif;
`ligandmpnn_fix_catalytic(...)` (mock) designs sequences with the triad fixed; `make_dead_mutant(...)`
builds the **catalytic-Ser→Ala negative control** (same fold, no nucleophile — the perfect negative).
**Every number here is SYNTHETIC** — this only proves the plumbing runs anywhere. Switch to the real
backends (RFdiffusion2/Riff-Diff on an A100; LigandMPNN CPU-fast) in `02_generate.ipynb`.

In [ ]:
from enzyme_tools import (scaffold_motif, ligandmpnn_fix_catalytic,
                          catalytic_geometry_rmsd, make_dead_mutant)

scaffolds = scaffold_motif(theo, n=5, method="mock")
print(f"{len(scaffolds)} mock scaffolds; example:")
print(" ", scaffolds[0])

seqs = ligandmpnn_fix_catalytic(scaffolds[0], theo.catalytic_residue_ids(), n=3, tool="mock")
print(f"\n{len(seqs)} mock sequences for {scaffolds[0]['design_id']} "
      f"(fixed roles: {seqs[0]['fixed_catalytic_roles']})")

# Catalytic-Ser->Ala dead mutant (the perfect negative control): pick a Ser in the
# (synthetic) sequence; in a real design this index is the DESIGNED catalytic serine.
seq0 = seqs[0]["sequence"]
ser_idx = seq0.find("S")
dead = make_dead_mutant(seq0, ser_idx)
n_diff = sum(a != b for a, b in zip(seq0, dead))
print(f"\ndead mutant: pos {ser_idx} S->A; differs at exactly {n_diff} position "
      f"(perfect negative control)")

cg = catalytic_geometry_rmsd(None, theo)   # mock, SYNTHETIC
print(f"\ncatalytic_geometry_rmsd (mock, SYNTHETIC) = {cg} A  -> pass if < 0.5 A")
print("NOTE: these are placeholder numbers. The real campaign is in notebook 02.")

## The metrics that decide an enzyme design
| Metric | Cutoff (`enzyme`) | Means | Does **not** mean |
|--------|-------------------|-------|-------------------|
| scRMSD | ≤ 2.0 Å | designed-vs-predicted backbone self-consistency | activity |
| pLDDT (global) | ≥ 85 | local fold confidence | thermostability / catalysis |
| pLDDT (catalytic) | ≥ 90 | confidence *at the active site* | the geometry is correct |
| **catalytic_geom_rmsd** | **< 0.5 Å** | predicted triad + oxyanion-hole atoms vs the theozyme | **activity** (can still be dead) |

Project-specific orthogonal checks (notebook 04): **pocket accessibility** (the ester docks
oriented toward Ser-OG) and **substrate scope** (which acyl chains fit). The fourth row is the
point of the whole project — and the last column is the message to never forget: **in-silico
catalytic geometry does not guarantee a working enzyme.** Only a kinetic assay does (notebook 05).

## D0 checklist
- [ ] Half-page on de novo enzyme design + the honest serine-hydrolase hit-rate reality.
- [ ] 1-page problem statement with **measurable** success criteria + the controls you'll need.
- [ ] Theozyme spec started in `data/inputs/theozyme_def.txt` (triad + **oxyanion hole**; cite sources).
- [ ] Reproduced mock hello-world (triad + oxyanion-hole spec + a mock scaffold + the dead-mutant control).
- [ ] `LOG.md` entry (tool versions, GPU, seed).

**Next:** `02_generate.ipynb` — scaffold the motif and run LigandMPNN with the catalytic triad fixed.